# 🦜🔗 LangGraph 응용 강의 - 3강

이 노트북에서는 LangGraph의 세 가지 핵심 워크플로우 패턴을 학습합니다:

1. **Prompt Chaining** — 여러 단계를 순차적으로 연결하는 파이프라인
2. **Parallelization** — 여러 작업을 동시에 병렬로 처리하는 패턴
3. **Routing** — 조건에 따라 실행 경로를 동적으로 선택하는 패턴

> 📖 [LangGraph Workflow 공식문서](https://docs.langchain.com/oss/python/langgraph/workflows-agents#orchestrator-worker)

## 📦 패키지 설치

LangGraph와 LangChain 관련 필수 패키지를 설치합니다.

In [ ]:
# LangGraph, LangChain, Google Gemini, OpenAI 패키지 설치
# -q : 출력 최소화, -U : 최신 버전으로 업그레이드
!pip install -qU langgraph langchain langchain-google-genai langchain-openai

## 🔧 공통 임포트

세 가지 패턴 모두에서 공통으로 사용하는 라이브러리를 미리 임포트합니다.

In [ ]:
# LangGraph의 핵심 구성요소 임포트
from langgraph.graph import StateGraph, START, END  # 그래프 생성 및 시작/끝 노드

# LangChain의 채팅 모델 초기화 유틸리티
from langchain.chat_models import init_chat_model

# Python 타입 힌팅: 상태(State) 딕셔너리 정의에 사용
from typing import TypedDict

# 데이터 유효성 검사를 위한 Pydantic (Routing 패턴에서 구조화 출력에 사용)
from pydantic import BaseModel, Field

# 불필요한 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

print("✅ 모든 라이브러리 임포트 완료!")

---

# 1️⃣ Prompt Chaining (프롬프트 체이닝)

## 개념 설명

**Prompt Chaining**은 여러 LLM 호출을 **순차적으로** 연결하는 패턴입니다.

- 이전 단계의 출력이 다음 단계의 입력이 됩니다.
- 복잡한 작업을 작은 단계로 분리하여 품질을 높입니다.
- 각 단계별로 다른 프롬프트/지시사항을 줄 수 있습니다.

```
START → [1단계: 초안 생성] → [2단계: 수정/개선] → [3단계: 최종 포장] → END
```

### 이번 예제: 3단계 농담 생성기
1. 주제로 농담 초안 만들기
2. 아재개그 스타일로 개선하기
3. 이모지를 추가해 SNS용으로 꾸미기

## 🤖 Model 정의

In [ ]:
# LLM 모델 초기화
# - "gpt-4o-mini": OpenAI의 경량 모델 (빠르고 비용 효율적)
# - temperature=1.0: 창의성 최대 (0에 가까울수록 일관적, 1에 가까울수록 창의적)
model = init_chat_model("gpt-4o-mini", temperature=1.0)

print(f"✅ 모델 초기화 완료: {model.model_name}")

## 📊 State 정의

**State(상태)**는 그래프의 모든 노드가 공유하는 데이터 저장소입니다.

- 각 노드는 State를 읽고, 자신의 결과를 State에 저장합니다.
- `TypedDict`를 사용하면 각 키의 타입을 명시할 수 있어 안전합니다.

In [ ]:
# 농담 생성 파이프라인의 상태(State) 정의
# 각 필드는 파이프라인의 단계별 결과를 저장합니다.
class JokeState(TypedDict):
    topic: str          # 입력: 농담의 주제 (예: "고양이", "프로그래머")
    draft_joke: str     # 1단계 결과: AI가 생성한 농담 초안
    improved_joke: str  # 2단계 결과: 아재개그 스타일로 개선된 버전
    final_joke: str     # 3단계 결과: 이모지가 추가된 최종 완성본

## 🔲 Node 정의

각 노드(Node)는 State를 입력받아 처리 후 결과를 딕셔너리로 반환합니다.

> 💡 **핵심**: 노드 함수의 반환값은 State를 **업데이트할 키-값 쌍**입니다. 반환하지 않은 키는 그대로 유지됩니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Step 1] 농담 초안 생성 노드
# 역할: State에서 'topic'을 읽어 농담 초안을 LLM으로 생성합니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def generate_joke(state: JokeState):
    """
    1단계: 주어진 주제로 농담 초안을 생성합니다.
    
    Args:
        state (JokeState): topic 필드가 담긴 현재 상태
    
    Returns:
        dict: draft_joke 키에 생성된 초안을 담아 반환
    """
    topic = state["topic"]  # State에서 주제 추출
    print(f"\n--- [1단계] '{topic}' 주제로 농담 초안 생성 중 ---")

    # LLM에게 짧고 재미있는 농담 하나만 생성하도록 요청
    msg = model.invoke(f"'{topic}'에 대한 짧고 재미있는 농담을 하나 만들어줘. 다른 말은 하지 말고 농담 하나만 해")
    
    # 결과를 'draft_joke' 키로 State에 저장
    return {"draft_joke": msg.content}

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Step 2] 농담 개선 노드 (아재개그 스타일)
# 역할: 1단계에서 생성된 초안을 더 썰렁한 아재개그로 업그레이드합니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def critique_and_improve(state: JokeState):
    """
    2단계: 초안을 아재개그 스타일로 개선합니다.
    
    Args:
        state (JokeState): draft_joke 필드가 담긴 현재 상태
    
    Returns:
        dict: improved_joke 키에 개선된 버전을 담아 반환
    """
    original_joke = state["draft_joke"]  # 이전 단계의 결과(초안) 가져오기
    print(f"\n--- [2단계] 더 웃기게 수정 중 (아재개그 스타일) ---")

    # 초안을 기반으로 아재개그 스타일 개선 요청
    prompt = f"""
    다음 농담을 보고, 더 썰렁하고 재미있는 '아재개그' 스타일로 개선해줘. 다른 말은 하지 말고 하나만 답변해.
    원문: {original_joke}
    """
    msg = model.invoke(prompt)
    
    # 개선된 버전을 'improved_joke' 키로 State에 저장
    return {"improved_joke": msg.content}

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Step 3] 최종 포장 노드 (SNS용 꾸미기)
# 역할: 개선된 농담에 이모지를 추가해 SNS에 올리기 좋게 완성합니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def polish_joke(state: JokeState):
    """
    3단계: 이모지를 추가하여 SNS용으로 최종 완성합니다.
    
    Args:
        state (JokeState): improved_joke 필드가 담긴 현재 상태
    
    Returns:
        dict: final_joke 키에 완성된 최종본을 담아 반환
    """
    improved_joke = state["improved_joke"]  # 2단계 결과 가져오기
    print(f"\n--- [3단계] 이모지 추가 및 마무리 ---")

    # 이모지를 추가해 SNS용으로 꾸미기 요청
    prompt = f"""
    다음 농담에 적절한 이모지를 듬뿍 넣어서 SNS에 올리기 좋게 꾸며줘.
    농담: {improved_joke}
    """
    msg = model.invoke(prompt)
    
    # 최종 완성본을 'final_joke' 키로 State에 저장
    return {"final_joke": msg.content}

## 🗺️ 그래프 생성 및 컴파일

노드들을 등록하고 엣지(Edge)로 연결하여 그래프를 완성합니다.

```
START → generate_joke → critique_and_improve → polish_joke → END
```

이 구조를 **Linear Chain(선형 체인)**이라고 합니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 그래프 생성 및 구성
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1. StateGraph 객체 생성 - JokeState를 상태 스키마로 사용
app = StateGraph(JokeState)

# 2. 노드 등록 - (노드 이름, 실행할 함수)
app.add_node('generate_joke', generate_joke)          # 1단계: 초안 생성
app.add_node('critique_and_improve', critique_and_improve)  # 2단계: 개선
app.add_node('polish_joke', polish_joke)              # 3단계: 마무리

# 3. 엣지 연결 - 순서대로 일직선(Linear Chain)으로 연결
# START는 그래프의 진입점 (특별한 예약 노드)
app.add_edge(START, 'generate_joke')                  # 시작 → 1단계
app.add_edge('generate_joke', 'critique_and_improve') # 1단계 → 2단계
app.add_edge('critique_and_improve', 'polish_joke')   # 2단계 → 3단계
app.add_edge('polish_joke', END)                      # 3단계 → 종료

# 4. 컴파일 - 그래프를 실행 가능한 상태로 변환
chain = app.compile()

print("✅ Prompt Chaining 그래프 컴파일 완료!")

In [ ]:
# 그래프 구조 시각화 (Jupyter에서 이미지로 표시됩니다)
chain

## ▶️ 실행

`chain.invoke()`에 초기 State를 딕셔너리로 전달하면 그래프가 실행됩니다.

In [ ]:
# 그래프 실행: 'topic' 키에 주제를 넣어 전달
# invoke()는 그래프가 END에 도달할 때까지 실행되고 최종 State를 반환합니다
inputs = {"topic": "고양이"}
result = chain.invoke(inputs)

print("\n✅ 파이프라인 실행 완료!")

In [ ]:
# 최종 State 전체 출력 (모든 단계의 결과가 포함됨)
result

In [ ]:
# 단계별 결과를 보기 좋게 출력
print(f"📌 주제: {result['topic']}")
print(f"\n📝 1차 초안:\n{result['draft_joke']}")
print(f"\n😄 2차 수정 (아재개그 버전):\n{result['improved_joke']}")
print("\n" + "-" * 40)
print(f"🎉 최종 완성본 (SNS용):")
print(result['final_joke'])

---

# 2️⃣ Parallelization (병렬 처리)

## 개념 설명

**Parallelization**은 여러 작업을 **동시에 병렬로** 실행하는 패턴입니다.

- 서로 독립적인 작업을 동시에 실행하여 처리 시간을 줄입니다.
- **Fan-out**: START에서 여러 노드로 동시에 분기
- **Fan-in**: 여러 노드의 결과를 하나의 노드로 합침

```
              ┌→ [Worker A: 시 작성] ─────┐
START → Router ┼→ [Worker B: 소설 작성] ──┼→ [Aggregator: 취합] → END
              └→ [Worker C: 농담 작성] ──┘
```

### 이번 예제: 3가지 글쓰기 동시 실행
하나의 주제로 시, 소설, 농담을 동시에 생성하고 하나의 리포트로 합칩니다.

## 🤖 Model 정의

In [ ]:
# 병렬 처리 예제용 모델 초기화
# temperature를 지정하지 않으면 기본값(1.0) 사용
model = init_chat_model("gpt-4o-mini")

print(f"✅ 모델 초기화 완료: {model.model_name}")

## 📊 State 정의

In [ ]:
# 병렬 글쓰기 파이프라인의 상태(State) 정의
# 각 Worker가 결과를 각자의 필드에 독립적으로 저장합니다.
class WriterState(TypedDict):
    topic: str          # 입력: 글쓰기 주제
    poem: str           # Worker A 결과: 시(Poem)
    story: str          # Worker B 결과: 소설(Story)
    joke: str           # Worker C 결과: 농담(Joke)
    final_report: str   # Aggregator 결과: 최종 편집본 (세 결과 합산)

## 🔲 Node 정의

병렬로 실행될 **Worker 노드** 3개와 결과를 취합할 **Aggregator 노드** 1개를 정의합니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Worker A] 시인 노드 - 주제로 시(Poem) 작성
# 다른 Worker와 병렬로 동시에 실행됩니다!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def write_poem(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker A] '{topic}' 주제로 시(Poem) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 아름다운 시를 짧게 써줘.")
    return {"poem": msg.content}  # 'poem' 필드만 업데이트

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Worker B] 소설가 노드 - 주제로 짧은 소설(Story) 작성
# Worker A, C와 동시에 병렬 실행됩니다!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def write_story(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker B] '{topic}' 주제로 소설(Story) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 감동적인 짧은 이야기를 써줘.")
    return {"story": msg.content}  # 'story' 필드만 업데이트

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Worker C] 개그맨 노드 - 주제로 아재개그(Joke) 작성
# Worker A, B와 동시에 병렬 실행됩니다!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def write_joke(state: WriterState):
    topic = state["topic"]
    print(f"   [Worker C] '{topic}' 주제로 농담(Joke) 작성 시작...")

    msg = model.invoke(f"'{topic}'에 대한 재미있는 아재개그를 하나 해줘.")
    return {"joke": msg.content}  # 'joke' 필드만 업데이트

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Aggregator] 편집장 노드 - 모든 Worker의 결과를 취합
# ⚠️ 중요: 모든 Worker가 완료된 후에만 실행됩니다! (Fan-in)
# LangGraph가 자동으로 모든 Worker 완료를 기다립니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def aggregator(state: WriterState):
    print("\n--- [Aggregator] 모든 원고 도착! 최종 편집 중 ---")

    # State에 저장된 각 Worker의 결과물을 꺼내서 하나로 합침
    final_text = f"""
    [주제: {state['topic']} 종합 선물세트]

    1. 시 (Poem)
    ----------------
    {state['poem']}

    2. 소설 (Story)
    ----------------
    {state['story']}

    3. 농담 (Joke)
    ----------------
    {state['joke']}
    """
    return {"final_report": final_text}  # 'final_report' 필드에 통합 결과 저장

## 🗺️ 그래프 생성 및 컴파일

Fan-out (1→여럿) 과 Fan-in (여럿→1) 구조를 엣지로 표현합니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 병렬 처리 그래프 생성
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
workflow = StateGraph(WriterState)

# 노드 등록 (4개: Worker 3개 + Aggregator 1개)
workflow.add_node('write_poem', write_poem)
workflow.add_node('write_story', write_story)
workflow.add_node('write_joke', write_joke)
workflow.add_node('aggregator', aggregator)

# ── Fan-out 엣지: START에서 3개의 Worker로 동시에 분기 ──
# LangGraph가 자동으로 병렬 실행을 처리합니다!
workflow.add_edge(START, 'write_poem')   # START → Worker A (시 작성)
workflow.add_edge(START, 'write_story')  # START → Worker B (소설 작성)
workflow.add_edge(START, 'write_joke')   # START → Worker C (농담 작성)

# ── Fan-in 엣지: 3개의 Worker에서 Aggregator 하나로 합침 ──
# LangGraph가 자동으로 모든 Worker가 완료되길 기다립니다!
workflow.add_edge('write_poem', 'aggregator')
workflow.add_edge('write_story', 'aggregator')
workflow.add_edge('write_joke', 'aggregator')

# Aggregator 완료 후 종료
workflow.add_edge('aggregator', END)

# 컴파일
app = workflow.compile()

print("✅ Parallelization 그래프 컴파일 완료!")

In [ ]:
# 그래프 구조 시각화 - Fan-out과 Fan-in 구조가 보입니다
app

## ▶️ 실행

In [ ]:
# 병렬 그래프 실행
# 3개의 Worker가 동시에 실행되므로, 순차 실행보다 훨씬 빠릅니다!
inputs = {"topic": "직장인의 월요일"}
result = app.invoke(inputs)

print("\n✅ 병렬 실행 완료!")

In [ ]:
# 원시 결과 출력 (State 딕셔너리 전체)
result

In [ ]:
# 최종 리포트만 보기 좋게 출력
print(result["final_report"])

---

# 3️⃣ Routing (라우팅)

## 개념 설명

**Routing**은 입력의 내용에 따라 **동적으로 실행 경로를 선택**하는 패턴입니다.

- **조건부 엣지(Conditional Edge)**: 함수의 반환값에 따라 다음 노드를 결정합니다.
- 다양한 유형의 입력을 적절한 전문가에게 자동으로 연결합니다.

```
               ┌─ 결제 문의 → [Billing Expert]
               ├─ 기술 문의 → [Tech Expert]
START → Router ┤
               ├─ 배송 문의 → [Shipping Expert]
               └─ 기타 문의 → [General Expert]
```

### 이번 예제: 고객 센터 자동 라우팅 시스템
고객 문의 내용을 LLM이 분석하여 적합한 담당 부서로 자동 연결합니다.

## 🤖 Model 정의

In [ ]:
# 라우팅 예제용 모델 초기화
model = init_chat_model("gpt-4o-mini")

print(f"✅ 모델 초기화 완료: {model.model_name}")

## 📊 State 정의

In [ ]:
# 고객 지원 시스템의 상태(State) 정의
class SupportState(TypedDict):
    query: str      # 입력: 고객의 문의 내용
    category: str   # Router가 결정한 카테고리 (예: 'billing', 'technical')
    response: str   # 전문가 노드가 생성한 최종 답변

## 🏷️ 구조화 출력(Structured Output) 정의

LLM이 자유 텍스트 대신 **정해진 형식**으로만 응답하도록 강제합니다.

- `Pydantic BaseModel`로 출력 스키마를 정의합니다.
- `Literal` 타입으로 가능한 값을 제한합니다.
- LangChain의 `with_structured_output()`으로 적용합니다.

In [ ]:
from typing import Literal  # 타입 힌팅: 정해진 값 중 하나만 허용

# 라우터의 분류 결과 스키마 정의
# LLM은 이 모델의 형식으로만 응답할 수 있습니다.
class RouteDecision(BaseModel):
    # Literal: 아래 4개의 값 중 하나만 선택 가능
    category: Literal['billing', 'technical', 'shipping', 'general'] = Field(
        description='고객 문의 내용을 분석하여 적절한 부서를 선택하세요.'
    )

# 구조화 출력을 사용하는 특수 라우터 LLM 생성
# 이 LLM은 RouteDecision 형식으로만 응답합니다!
router_llm = model.with_structured_output(RouteDecision)

print("✅ 라우터 LLM 설정 완료!")
print(f"   가능한 카테고리: billing, technical, shipping, general")

## 🔲 Node 정의

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# [Manager] 라우터 노드 - 문의 내용을 분석하여 카테고리 결정
# LLM이 Rule 기반이 아닌 문맥을 이해하여 동적으로 분류합니다!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def router_node(state: SupportState):
    """
    고객 문의를 분석하여 적절한 부서(카테고리)를 결정합니다.
    
    구조화 출력(Structured Output)을 사용하여
    LLM이 정확히 4개의 카테고리 중 하나만 선택하도록 강제합니다.
    """
    query = state["query"]
    print(f"\n--- [Router] 문의 분석 중: '{query}' ---")

    # router_llm은 RouteDecision 형식으로만 응답 → category 속성 추출 가능!
    decision = router_llm.invoke(query)

    print(f"   -> 분류 결과: {decision.category.upper()} 부서로 연결합니다.")
    return {"category": decision.category}  # 다음 노드 선택에 사용될 카테고리 저장

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 각 부서별 전문가 노드 정의 (4개)
# 라우터가 결정한 카테고리에 따라 그 중 하나만 실행됩니다.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# [Expert A] 결제/환불 담당
def billing_expert(state: SupportState):
    print("--- [Billing Expert] 결제 전문가가 답변 작성 중 ---")
    prompt = f"당신은 결제 및 환불 전문가입니다. 다음 문의에 대해 정중하게 답변하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert B] 기술 지원 담당
def technical_expert(state: SupportState):
    print("--- [Tech Expert] 기술 지원 엔지니어가 분석 중 ---")
    prompt = f"당신은 IT 엔지니어입니다. 다음 기술 문제에 대해 해결책을 제시하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert C] 배송 담당
def shipping_expert(state: SupportState):
    print("--- [Shipping Expert] 물류 담당자가 배송 조회 중 ---")
    prompt = f"당신은 배송 관리자입니다. 다음 배송 문의에 대해 답변하세요: {state['query']}"
    msg = model.invoke(prompt)
    return {"response": msg.content}

# [Expert D] 일반 상담
def general_expert(state: SupportState):
    print("--- [General] 일반 상담원이 답변 중 ---")
    msg = model.invoke(f"다음 문의에 친절하게 답변하세요: {state['query']}")
    return {"response": msg.content}

print("✅ 전문가 노드 4개 정의 완료!")

## 🗺️ 그래프 생성 및 컴파일

**조건부 엣지(Conditional Edge)**를 사용하여 라우터의 결정에 따라 경로를 분기합니다.

In [ ]:
# 그래프 생성
workflow = StateGraph(SupportState)

# 모든 노드 등록 (라우터 1개 + 전문가 4개)
workflow.add_node("router_node", router_node)
workflow.add_node("billing_expert", billing_expert)
workflow.add_node("technical_expert", technical_expert)
workflow.add_node("shipping_expert", shipping_expert)
workflow.add_node("general_expert", general_expert)

# START → 라우터 (항상 라우터부터 시작)
workflow.add_edge(START, "router_node")

print("✅ 노드 및 기본 엣지 등록 완료!")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 조건부 엣지 라우팅 함수 정의
# 이 함수가 반환하는 값(문자열)이 다음 실행할 노드의 이름이 됩니다!
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def route_to_expert(state: SupportState):
    """
    State의 'category' 값을 읽어 실행할 다음 노드 이름을 반환합니다.
    
    Returns:
        str: 다음에 실행할 노드의 이름
    """
    category = state['category']

    # 카테고리에 따라 적절한 전문가 노드로 라우팅
    if category == 'billing':
        return 'billing_expert'    # 결제 문의 → 결제 전문가
    elif category == 'technical':
        return 'technical_expert'  # 기술 문의 → 기술 전문가
    elif category == 'shipping':
        return 'shipping_expert'   # 배송 문의 → 배송 담당자
    else:
        return 'general_expert'    # 그 외 → 일반 상담원

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 조건부 엣지 등록
# add_conditional_edges(시작노드, 라우팅함수, [가능한 다음 노드 목록])
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
workflow.add_conditional_edges(
    'router_node',        # 이 노드 실행 후 조건부 분기
    route_to_expert,      # 분기를 결정하는 함수
    # 가능한 다음 노드 목록 (명시적으로 선언하는 것이 좋은 관행)
    ['billing_expert', 'technical_expert', 'shipping_expert', 'general_expert']
)

print("✅ 조건부 엣지 등록 완료!")

In [ ]:
# 각 전문가 노드 → END 엣지 연결 (모든 전문가 답변 후 종료)
workflow.add_edge('billing_expert', END)
workflow.add_edge('technical_expert', END)
workflow.add_edge('shipping_expert', END)
workflow.add_edge('general_expert', END)

# 컴파일
app = workflow.compile()

print("✅ Routing 그래프 컴파일 완료!")

In [ ]:
# 그래프 구조 시각화 - 조건부 분기 구조가 보입니다
app

## ▶️ 실행 - 다양한 고객 문의 테스트

서로 다른 유형의 문의를 입력하여 자동으로 올바른 부서로 라우팅되는지 확인합니다.

In [ ]:
# ─── 테스트 1: 결제 관련 문의 ───
# 예상 라우팅: billing 부서
print("="*50)
print("[테스트 1] 결제 관련 문의")
print("="*50)

result_billing = app.invoke({"query": "지난달 요금이 두 번 빠져나갔어요. 확인해주세요."})

print(f"\n📂 라우팅 카테고리: {result_billing['category']}")
print(f"💬 답변 요약: {result_billing['response'][:200]}...")

In [ ]:
# ─── 테스트 2: 기술 문제 문의 ───
# 예상 라우팅: technical 부서
print("="*50)
print("[테스트 2] 기술 문제 문의")
print("="*50)

result_tech = app.invoke({"query": "API 연결할 때 404 에러가 자꾸 떠요."})

print(f"\n📂 라우팅 카테고리: {result_tech['category']}")
print(f"💬 답변 요약: {result_tech['response'][:200]}...")

In [ ]:
# ─── 테스트 3: 배송 관련 문의 ───
# 예상 라우팅: shipping 부서
print("="*50)
print("[테스트 3] 배송 관련 문의")
print("="*50)

result_shipping = app.invoke({"query": "주문한 노트북 언제 도착하나요?"})

print(f"\n📂 라우팅 카테고리: {result_shipping['category']}")
print(f"💬 답변 요약: {result_shipping['response'][:200]}...")

---

# 📝 강의 요약

## LangGraph 3가지 핵심 패턴 비교

| 패턴 | 구조 | 장점 | 사용 사례 |
|------|------|------|----------|
| **Prompt Chaining** | A → B → C | 단계적 품질 향상 | 초안 → 검토 → 최종본 |
| **Parallelization** | A, B, C 동시 → D | 처리 시간 단축 | 다각도 분석, 다중 생성 |
| **Routing** | 조건 → A 또는 B 또는 C | 적재적소 처리 | 고객 지원, 분류 시스템 |

## 핵심 개념 복습

- **State**: 그래프 전체에서 공유되는 데이터 저장소 (`TypedDict`로 정의)
- **Node**: 상태를 처리하는 함수 (입력: State, 출력: 업데이트할 딕셔너리)
- **Edge**: 노드 간 연결 (일반 엣지 vs 조건부 엣지)
- **Fan-out**: START → 여러 노드 (병렬 실행)
- **Fan-in**: 여러 노드 → 하나의 노드 (결과 취합)
- **Structured Output**: LLM 출력을 정해진 형식으로 강제